In [ ]:
! curl -L https://raw.githubusercontent.com/karpathy/ng-video-lecture/master/input.txt -o input.txt

^C


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0           00:01              0
  5  1.06M   5  65536   0      0  29636      0   00:37   00:02   00:35  61368
 13  1.06M  13 144.0k   0      0  44783      0   00:24   00:03   00:21  68607
 19  1.06M  19 208.0k   0      0  47428      0   00:23   00:04   00:19  63629
 23  1.06M  23 256.0k   0      0  47046      0   00:23   00:05   00:18  59193
 27  1.06M  27 304.0k   0      0  47134      0   00:23   00:06   00:17  57004
 30  1.06M  30 336.0k   0      0  44050      0   00:25   00:07   00:18  49742
 33  1.06M  33 368.0k   0      0  41382      0   00:26   00:09   00:17  39456
 38  1.06M  38 416.0k   0      0  42059      0   00:26   00:10   00:16  37782
 42  1.06M  42 464.0k   0      0  41609      0   00:26   00:11   00:15  36428
 45  1.06M  45 496.0k   0      0  40352      0   00:27   00:12 

In [3]:
with open("input.txt") as f:
    text = f.read()

len(text),print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



(344064, None)

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !&',-.:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
63


In [5]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s] # encoder: take a  string, output a list of inegers
decode = lambda l: [''.join(itos[c] for c in l)] # decoder: take a list of integers, output a string
print(encode("hi there"))
print(decode(encode("hi there")))

[44, 45, 1, 56, 44, 41, 54, 41]
['hi there']


there could be many possible ways to do encoding and decoding you can also look at the *SentencePiece* tokenizer and also the **tik tokenizer** thats what the initail models of chat gpt uses

In [6]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:1000])

torch.Size([344064]) torch.int64
tensor([16, 45, 54, 55, 56,  1, 13, 45, 56, 45, 62, 41, 50,  8,  0, 12, 41, 42,
        51, 54, 41,  1, 59, 41,  1, 52, 54, 51, 39, 41, 41, 40,  1, 37, 50, 61,
         1, 42, 57, 54, 56, 44, 41, 54,  5,  1, 44, 41, 37, 54,  1, 49, 41,  1,
        55, 52, 41, 37, 47,  7,  0,  0, 11, 48, 48,  8,  0, 29, 52, 41, 37, 47,
         5,  1, 55, 52, 41, 37, 47,  7,  0,  0, 16, 45, 54, 55, 56,  1, 13, 45,
        56, 45, 62, 41, 50,  8,  0, 35, 51, 57,  1, 37, 54, 41,  1, 37, 48, 48,
         1, 54, 41, 55, 51, 48, 58, 41, 40,  1, 54, 37, 56, 44, 41, 54,  1, 56,
        51,  1, 40, 45, 41,  1, 56, 44, 37, 50,  1, 56, 51,  1, 42, 37, 49, 45,
        55, 44, 10,  0,  0, 11, 48, 48,  8,  0, 28, 41, 55, 51, 48, 58, 41, 40,
         7,  1, 54, 41, 55, 51, 48, 58, 41, 40,  7,  0,  0, 16, 45, 54, 55, 56,
         1, 13, 45, 56, 45, 62, 41, 50,  8,  0, 16, 45, 54, 55, 56,  5,  1, 61,
        51, 57,  1, 47, 50, 51, 59,  1, 13, 37, 45, 57, 55,  1, 23, 37, 54, 39,
       

In [7]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
train_data.shape,val_data.shape

(torch.Size([309657]), torch.Size([34407]))

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([16, 45, 54, 55, 56,  1, 13, 45, 56])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([16]) the target: 45
when input is tensor([16, 45]) the target: 54
when input is tensor([16, 45, 54]) the target: 55
when input is tensor([16, 45, 54, 55]) the target: 56
when input is tensor([16, 45, 54, 55, 56]) the target: 1
when input is tensor([16, 45, 54, 55, 56,  1]) the target: 13
when input is tensor([16, 45, 54, 55, 56,  1, 13]) the target: 45
when input is tensor([16, 45, 54, 55, 56,  1, 13, 45]) the target: 56


In [10]:
torch.manual_seed(1337)
batch_size = 4 #  how many independen sequences will be process in parallel?
block_size = 8 # what is the maximum context lenght for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb = get_batch("train")
print("inputs")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")
    

inputs
torch.Size([4, 8])
tensor([[30, 44, 41, 55, 41,  1, 37, 54],
        [45, 50, 42, 45, 40, 41, 48, 55],
        [41,  1, 48, 41, 43,  5,  1, 56],
        [44, 41,  1, 47, 45, 50, 43,  4]])
targets:
torch.Size([4, 8])
tensor([[44, 41, 55, 41,  1, 37, 54, 41],
        [50, 42, 45, 40, 41, 48, 55, 10],
        [ 1, 48, 41, 43,  5,  1, 56, 44],
        [41,  1, 47, 45, 50, 43,  4, 55]])
----
when input is [30] the target: 44
when input is [30, 44] the target: 41
when input is [30, 44, 41] the target: 55
when input is [30, 44, 41, 55] the target: 41
when input is [30, 44, 41, 55, 41] the target: 1
when input is [30, 44, 41, 55, 41, 1] the target: 37
when input is [30, 44, 41, 55, 41, 1, 37] the target: 54
when input is [30, 44, 41, 55, 41, 1, 37, 54] the target: 41
when input is [45] the target: 50
when input is [45, 50] the target: 42
when input is [45, 50, 42] the target: 45
when input is [45, 50, 42, 45] the target: 40
when input is [45, 50, 42, 45, 40] the target: 41
when input is

In [ ]:
max_iter = 3000
eval_interval = 300
learning_rate = 1e-2
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
n_embd = 32

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token directly reads off the logtis for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        self.lm_head = nn.Linear(n_embd,vocab_size)

    def forward(self,idx,target=None):
        B,T = idx.shape
        #idx and targets are both (B,T) tensors of integers
        tok_embd = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T,device=device)) # (T,C)
        x = tok_embd + pos_emb # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if target is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            target = target.view(B*T)
            loss = F.cross_entropy(logits,target)
        
        return logits, loss

    def generate(self,idx,max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits,loss = self(idx)
            # focus only on the last time step
            logits = logits[:,-1,:] # Becomes (B,C)
            # apply softmax to get probabilities 
            probs = F.softmax(logits,dim=1) # (B,C)
            # sample from the distribution
            idx_next = torch.multinomial(probs,num_samples=1) # (B,1)
            idx = torch.cat((idx,idx_next),dim=1) # (B,T+1)
        return idx

m = BigramLanguageModel()
logits,loss = m(xb,yb)
logits.shape,loss.item()

(torch.Size([32, 65]), 4.878634929656982)

In [ ]:
decode(m.generate(torch.zeros((1,1),dtype=torch.long),100)[0].tolist())

["\nSr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3"]

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000):
    # sample a batch data
    xb,yb = get_batch("train")

    logits,loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5727508068084717


In [ ]:
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),400)[0].tolist()))

["\nIyoteng h hasbe pave pirance\nRie hicomyonthar's\nPlinseard ith henoure wounonthioneir thondy, y heltieiengerofo'dsssit ey\nKIN d pe wither vouprrouthercc.\nhathe; d!\nMy hind tt hinig t ouchos tes; st yo hind wotte grotonear 'so it t jod weancotha:\nh hay.JUCle n prids, r loncave w hollular s O:\nHIs; ht anjx?\n\nDUThinqunt.\n\nLaZAnde.\nathave l.\nKEONH:\nARThanco be y,-hedarwnoddy scace, tridesar, wnl'shenou"]


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
xbox = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbox[b,t] = torch.mean(xprev,0)

In [ ]:
x[1,:4].mean(dim=0),x[1,:4]

(tensor([0.2711, 0.4774]),
 tensor([[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910]]))